# From-scratch OCR-VLM — training on Kaggle Notebooks (M1)

Trains the **from-scratch OCR-free receipt VLM** (`OcrVLM` = CNN-stem encoder + transformer
decoder, no CLIP / no SmolLM2) on on-the-fly multilingual synthetic receipts. Same resumable /
durable structure as `train_receipt_vlm_kaggle.ipynb`:

- **Every epoch** writes a resumable checkpoint (`ocr_vlm_epoch{NN}_loss{L}.pt`, model+optimizer)
  and **pushes it to a durable Kaggle *Dataset*** — a session timeout loses at most the current
  epoch. `train_ocr_vlm.py` **auto-resumes** from the latest checkpoint in the dir.
- A fixed tokenizer (`tokenizer.json`) keeps the vocab identical across sessions.

## One-time setup
1. **Rebuild + upload the code bundle** so it contains the new OCR-VLM code
   (`receipt_vlm/models/ocr_*.py`, `receipt_vlm/data/{lin_schema,tokenizer,ocr_transform,ocr_dataset}.py`,
   `scripts/train_ocr_vlm.py`): on your PC `python scripts/zip_selfcontained_colab.py`, then upload
   `colab_upload/receipt_vlm_colab_bundle.zip` as a Kaggle Dataset and *Add Input* here.
2. **Accelerator:** *Settings* → **GPU T4 x2** (or P100).
3. **Internet: ON** (phone-verified account; needed for pip + the dataset push).
4. **API token as Secrets:** Account → *Create New API Token*; then *Add-ons → Secrets* → add
   `KAGGLE_USERNAME` and `KAGGLE_KEY`.
5. For an unattended run: **Save Version → Save & Run All (Commit)** (background up to 12 h).

## 0. Configuration — edit then run

In [ ]:
# --- training (from-scratch OCR-VLM) ---
# TARGET: what the decoder emits.
#   "schema"        -> [STORE]..[ITEM]..[PRICE].. (terse, image-specific): teaches image-grounding
#                      FASTEST / most sample-efficient (proven in the overfit sanity). Directly usable.
#   "transcription" -> all visible text (Stage-A READ per the plan): richer read signal, but much of
#                      the target is boilerplate the decoder can language-model, so it needs LOTS of
#                      distinct samples (big N + epochs) before it's forced to actually read.
TARGET      = "transcription"  # switch to "schema" if READ is slow to ground on your compute budget
N_PER_EPOCH = 20000            # synthetic receipts generated per epoch (on-the-fly, no disk)
LANGUAGES   = "fr,en,es,de,it" # Latin-script locales to mix
EPOCHS      = 40
BATCH_SIZE  = 24
LR          = 3e-4
EMBED_DIM   = 256
ENC_DEPTH   = 4
DEC_DEPTH   = 4
HEADS       = 8
MAX_LEN     = 768              # transcription targets are long; keep high (>= your longest target)
DISTORT     = True             # apply capture distortions to synthetic (harder, more realistic)
INTENSITY   = "light"          # light | medium | heavy
NUM_WORKERS = 2
LOG_EVERY   = 200              # batches between heartbeat prints
EVAL_EVERY  = 1               # epochs between quick synthetic evals (0 = off)
KEEP_LAST   = 3                # prune to newest N epoch checkpoints (bounds dataset push size)

# --- durable checkpoint storage ---
SAVE_TO_DATASET  = True
SAVE_EVERY_EPOCH = True                     # push every epoch (max safety); False = once per run
DATASET_NAME     = "receipt-ocr-vlm-checkpoints"   # slug under YOUR account (lowercase + hyphens)

## 1. GPU check

In [ ]:
import torch
assert torch.cuda.is_available(), "Enable GPU: Settings -> Accelerator -> GPU T4 x2"
print(torch.cuda.get_device_name(0))

## 1b. Kaggle API auth + dataset helper
Reads your `KAGGLE_USERNAME` / `KAGGLE_KEY` Secrets and defines `kaggle_save()` (create-or-version
the checkpoint dataset). Checkpoint files upload individually so a later session resumes from them.

In [ ]:
import json, os, subprocess
from pathlib import Path

DATASET_ID = None

def _kaggle_auth():
    global DATASET_ID
    from kaggle_secrets import UserSecretsClient
    sec = UserSecretsClient()
    try:
        user = sec.get_secret("KAGGLE_USERNAME")
        key  = sec.get_secret("KAGGLE_KEY")
    except Exception as e:
        raise RuntimeError(
            "Missing Secrets. Add-ons -> Secrets -> add KAGGLE_USERNAME and KAGGLE_KEY "
            "(values from Account -> Create New API Token / kaggle.json)."
        ) from e
    kdir = Path.home() / ".kaggle"; kdir.mkdir(exist_ok=True)
    (kdir / "kaggle.json").write_text(json.dumps({"username": user, "key": key}))
    os.chmod(kdir / "kaggle.json", 0o600)
    os.environ["KAGGLE_USERNAME"], os.environ["KAGGLE_KEY"] = user, key
    DATASET_ID = f"{user}/{DATASET_NAME}"

def kaggle_save(folder, msg):
    """Create-or-version a Kaggle Dataset from `folder`. No-op if SAVE_TO_DATASET is False."""
    if not SAVE_TO_DATASET:
        return
    folder = Path(folder)
    if not [p for p in folder.glob("*.pt")]:
        return
    (folder / "dataset-metadata.json").write_text(json.dumps({
        "title": DATASET_NAME, "id": DATASET_ID, "licenses": [{"name": "CC0-1.0"}],
    }))
    exists = subprocess.run(["kaggle", "datasets", "files", DATASET_ID],
                            capture_output=True, text=True).returncode == 0
    sub = "version" if exists else "create"
    cmd = ["kaggle", "datasets", sub, "-p", str(folder), "--dir-mode", "skip"]
    if exists:
        cmd += ["-m", msg]
    print(f"   [dataset] {sub}: {msg}", flush=True)
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print("   [dataset] WARNING push failed:", r.stderr.strip()[:300], flush=True)
    else:
        print(f"   [dataset] -> https://www.kaggle.com/datasets/{DATASET_ID}", flush=True)

if SAVE_TO_DATASET:
    _kaggle_auth()
    print("Checkpoints -> dataset:", DATASET_ID, "| every epoch:", SAVE_EVERY_EPOCH)
else:
    print("SAVE_TO_DATASET=False -- /kaggle/working only (lost on interactive timeout unless you Commit)")

## 2. Get the code from your attached bundle
Copies the bundle into the writable `/kaggle/working` (editable installs can't be written under the
read-only `/kaggle/input`).

In [ ]:
import glob, os, shutil, zipfile
from pathlib import Path

WORK = Path("/kaggle/working/repo")

def materialize():
    zips = glob.glob("/kaggle/input/**/receipt_vlm_colab_bundle.zip", recursive=True)
    if zips:
        WORK.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zips[0]) as zf:
            zf.extractall(WORK)
        return
    hits = glob.glob("/kaggle/input/**/vlm_training/scripts/train_ocr_vlm.py", recursive=True)
    if hits:
        dev_ocr_src = Path(hits[0]).resolve().parents[2]
        dest = WORK / "dev_ocr"
        if not dest.exists():
            shutil.copytree(dev_ocr_src, dest)
        return
    raise FileNotFoundError("Bundle not found in /kaggle/input -- did you Add Input (rebuilt bundle)?")

if not list(WORK.glob("**/vlm_training/scripts/train_ocr_vlm.py")):
    materialize()

hits = glob.glob(str(WORK / "**/vlm_training/scripts/train_ocr_vlm.py"), recursive=True)
assert hits, ("train_ocr_vlm.py not found after materialize -- your uploaded bundle predates the "
              "OCR-VLM code; rebuild it with scripts/zip_selfcontained_colab.py and re-upload.")
TRAIN_PKG = Path(hits[0]).resolve().parents[1]
DEV_OCR = TRAIN_PKG.parent
os.chdir(TRAIN_PKG)
print("Train package:", TRAIN_PKG)

## 3. Install dependencies (~2-3 min)

In [ ]:
import subprocess, sys
def pip(*a):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *a])

pip("-r", "requirements-training.txt", "tokenizers>=0.22,<=0.23")
pip("-e", str(DEV_OCR))        # receipt_ocr
pip("-e", str(TRAIN_PKG))      # receipt_vlm
print("Install OK")

## 4. Checkpoint dir
Checkpoints (and `tokenizer.json`) go to `/kaggle/working/ocr_ckpts`; cell 5 mirrors new epoch
checkpoints to your Kaggle Dataset as they're written.

In [ ]:
from pathlib import Path
CKPT_DIR = Path("/kaggle/working/ocr_ckpts"); CKPT_DIR.mkdir(parents=True, exist_ok=True)
print("Checkpoints ->", CKPT_DIR)

## 4b. Resume across sessions
Attach your `receipt-ocr-vlm-checkpoints` dataset via **Add Input**, then run this cell: it copies
every `ocr_vlm_epoch*.pt` + `tokenizer.json` back into the checkpoint dir. `train_ocr_vlm.py`
auto-resumes from the latest epoch — no flags to flip.

In [ ]:
import glob, os, shutil
restored = []
for src in glob.glob("/kaggle/input/**/ocr_vlm_epoch*.pt", recursive=True) + \
           glob.glob("/kaggle/input/**/tokenizer.json", recursive=True):
    dest = CKPT_DIR / os.path.basename(src)
    if not dest.exists():
        shutil.copy(src, dest); restored.append(dest.name)
print(f"Restored {len(restored)} file(s):", sorted(restored)[-6:] or "nothing (fresh run)")

## 5. Train (resumable, per-epoch dataset push)
Runs `scripts/train_ocr_vlm.py`; it auto-resumes from cell 4b's checkpoints. With
`SAVE_EVERY_EPOCH=True`, each new `ocr_vlm_epoch*.pt` is pushed to the Dataset the moment it's
written, so a crash costs at most the current epoch. Live timestamp/gap heartbeat.

> Each epoch checkpoint bundles the optimizer (bigger than the adapter checkpoints); `KEEP_LAST`
> prunes old ones so the pushed folder stays bounded. Set `SAVE_EVERY_EPOCH=False` for one push
> per run if the uploads bite.

In [ ]:
import subprocess, sys, os, re, time, datetime

cmd = [sys.executable, "-u", "scripts/train_ocr_vlm.py",
       "--target", TARGET, "--checkpoint-dir", str(CKPT_DIR),
       "--n", str(N_PER_EPOCH), "--languages", LANGUAGES, "--epochs", str(EPOCHS),
       "--batch-size", str(BATCH_SIZE), "--lr", str(LR),
       "--embed-dim", str(EMBED_DIM), "--enc-depth", str(ENC_DEPTH), "--dec-depth", str(DEC_DEPTH),
       "--heads", str(HEADS), "--max-len", str(MAX_LEN), "--num-workers", str(NUM_WORKERS),
       "--log-every", str(LOG_EVERY), "--eval-every", str(EVAL_EVERY), "--keep-last", str(KEEP_LAST),
       "--distort-intensity", INTENSITY]
if DISTORT:
    cmd.append("--distort")
print(">>", " ".join(cmd), flush=True)

env = {**os.environ, "PYTHONUNBUFFERED": "1"}
start = last = time.time()
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
for line in proc.stdout:
    now = time.time(); gap = now - last; last = now
    ts = datetime.datetime.now().strftime("%H:%M:%S")
    print(f"[{ts} +{int(now-start):>5}s gap{gap:4.0f}s] {line}", end="", flush=True)
    if SAVE_TO_DATASET and SAVE_EVERY_EPOCH and "Checkpoint saved" in line and "_epoch" in line:
        m = re.search(r"(ocr_vlm_epoch\d+_loss[\d.]+\.pt)", line)
        kaggle_save(CKPT_DIR, m.group(1) if m else "epoch checkpoint")
proc.wait()
if proc.returncode != 0:
    raise RuntimeError(f"train_ocr_vlm.py failed (exit {proc.returncode})")
if SAVE_TO_DATASET and not SAVE_EVERY_EPOCH:
    kaggle_save(CKPT_DIR, "after run")
print("Training done")

## 6. Get your files
The latest checkpoint + `tokenizer.json` are durable in your **`receipt-ocr-vlm-checkpoints`
Dataset** and under `/kaggle/working/ocr_ckpts`. Download the newest `ocr_vlm_epoch*.pt` +
`tokenizer.json` to run/evaluate locally (`scripts/evaluate.py` / the OcrVLM loader).

In [ ]:
from pathlib import Path
print("Files in", CKPT_DIR, ":")
for p in sorted(Path(CKPT_DIR).glob("*")):
    if p.is_file():
        print(f"  {p.name:40} {p.stat().st_size/1e6:8.1f} MB")
if SAVE_TO_DATASET and DATASET_ID:
    print("\nDurable copy -> https://www.kaggle.com/datasets/" + DATASET_ID)